# **TFM DSMarket**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings("ignore") 

import holidays
import gc

In [2]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

# DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'

# df_daily = pd.read_csv(DATA_PATH + 'daily_calendar_with_events.csv')
# df_prices = pd.read_csv(DATA_PATH + 'item_prices.csv')
# df_sales = pd.read_csv(DATA_PATH + 'item_sales.csv')

In [4]:
df_daily = pd.read_csv("data_dsmarket/daily_calendar_with_events.csv")
df_prices = pd.read_csv('data_dsmarket/item_prices.csv')
df_sales = pd.read_csv('data_dsmarket/item_sales.csv')

In [5]:
def reduce_mem_usage(df, turn_cat = False, silence = True):
    """Itera sobre todo el dataset convirtiendo cada columna en el tipo más adecuado para ahorrar memoria
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe que se quiere reducir
    turn_cat : bool, optional
        Transformación de las columnas objeto o string a category, by default False
    Returns
    -------
    pd.DataFrame
        Dataframe optimizado
    """
    start_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    for col in df.columns:
        col_type = df[col].dtype
        
        # Saltar columnas que ya son category o datetime
        if str(col_type) in ('category', 'datetime64[ns]'):
            continue
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                # Desactivamos el casteo a float16 por los errores de precisión
                # https://github.com/pandas-dev/pandas/issues/34618
                # if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                #     df[col] = df[col].astype(np.float16)
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        if turn_cat:
            df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    if not silence:
        print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [6]:
df_daily  = reduce_mem_usage(df_daily, silence=False)
df_prices = reduce_mem_usage(df_prices, silence=False)
df_sales  = reduce_mem_usage(df_sales, silence=False)

Decreased by 17.5%
Decreased by 20.0%
Decreased by 78.7%


## **Exploración inicial**

### **Tabla Ventas**

In [7]:
print(f'{df_sales.shape[0]:,} filas x {df_sales.shape[1]:,} columnas')

30,490 filas x 1,920 columnas


In [8]:
display(df_sales.head())
# df_sales.describe(include= 'all')

,id,item,category,department,store,store_code,region,d_1,d_2,d_3,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [9]:
df_sales.isna().sum().sum()      # No hay nulos

np.int64(0)

In [10]:
df_sales.dtypes.head(10)

id            object
item          object
category      object
department    object
store         object
store_code    object
region        object
d_1            int16
d_2            int16
d_3            int16
dtype: object

In [11]:
df_sales.category.value_counts()

category
SUPERMARKET      14370
HOME_&_GARDEN    10470
ACCESORIES        5650
Name: count, dtype: int64

In [12]:
df_sales.department.value_counts().sort_index()

department
ACCESORIES_1       4160
ACCESORIES_2       1490
HOME_&_GARDEN_1    5320
HOME_&_GARDEN_2    5150
SUPERMARKET_1      2160
SUPERMARKET_2      3980
SUPERMARKET_3      8230
Name: count, dtype: int64

In [13]:
df_sales.store_code.value_counts()

store_code
NYC_1    3049
NYC_2    3049
NYC_3    3049
NYC_4    3049
BOS_1    3049
BOS_2    3049
BOS_3    3049
PHI_1    3049
PHI_2    3049
PHI_3    3049
Name: count, dtype: int64

In [14]:
df_sales.store.value_counts()

store
Greenwich_Village    3049
Harlem               3049
Tribeca              3049
Brooklyn             3049
South_End            3049
Roxbury              3049
Back_Bay             3049
Midtown_Village      3049
Yorktown             3049
Queen_Village        3049
Name: count, dtype: int64

In [15]:
df_sales.region.value_counts()

region
New York        12196
Boston           9147
Philadelphia     9147
Name: count, dtype: int64

In [16]:
df_sales.groupby('store')['item'].count()

store
Back_Bay             3049
Brooklyn             3049
Greenwich_Village    3049
Harlem               3049
Midtown_Village      3049
Queen_Village        3049
Roxbury              3049
South_End            3049
Tribeca              3049
Yorktown             3049
Name: item, dtype: int64

El dataset contiene **3.049 productos** organizados en **3 categorías** y **7 departamentos**, 
con ventas registradas en **10 tiendas** distribuidas en **3 regiones** (New York, Boston y Philadelphia). 

El catálogo de productos es **idéntico en todas las tiendas**, aunque New York tiene 4 tiendas frente 
a las 3 de Boston y Philadelphia.

### **Tabla Precios**

In [17]:
print(f'{df_prices.shape[0]:,} filas x {df_prices.shape[1]:,} columnas')

6,965,706 filas x 5 columnas


In [18]:
df_prices = df_prices.sort_values(by = 'yearweek')

In [19]:
df_prices.head()

,item,category,store_code,yearweek,sell_price
5539160,SUPERMARKET_3_702,SUPERMARKET,PHI_1,201105.00,3.94
6613947,HOME_&_GARDEN_2_445,HOME_&_GARDEN,PHI_3,201105.00,8.10
261069,HOME_&_GARDEN_2_041,HOME_&_GARDEN,NYC_1,201105.00,6.59
1516539,HOME_&_GARDEN_1_117,HOME_&_GARDEN,NYC_3,201105.00,11.21
563249,SUPERMARKET_3_188,SUPERMARKET,NYC_1,201105.00,2.38


In [20]:
df_prices.isna().sum()

item               0
category           0
store_code         0
yearweek      243920
sell_price         0
dtype: int64

In [21]:
df_prices.sell_price.value_counts()

sell_price
2.38     221088
3.58     217873
3.00     189541
1.20     150387
4.78     142487
          ...  
10.53         1
4.75          1
0.55          1
2.21          1
25.10         1
Name: count, Length: 1888, dtype: int64

In [22]:
df_prices.sell_price.describe()

count   6965706.00
mean          5.52
std           4.35
min           0.01
25%           2.62
50%           4.20
75%           7.18
max         134.15
Name: sell_price, dtype: float64

In [23]:
df_prices.item.value_counts()

item
SUPERMARKET_3_702      2870
HOME_&_GARDEN_1_234    2870
ACCESORIES_1_194       2870
SUPERMARKET_1_168      2870
SUPERMARKET_3_590      2870
                       ... 
HOME_&_GARDEN_1_308     652
HOME_&_GARDEN_1_159     633
HOME_&_GARDEN_1_242     610
SUPERMARKET_3_296       602
SUPERMARKET_2_379       543
Name: count, Length: 3049, dtype: int64

Los 3049 productos tienen distinto número de registros de precio —> no todos 
estuvieron disponibles en todas las tiendas durante todas las semanas

### **Tabla Eventos**

In [24]:
print(f'{df_daily.shape[0]:,} filas x {df_daily.shape[1]:,} columnas')

1,913 filas x 5 columnas


In [25]:
df_daily.columns

Index(['date', 'weekday', 'weekday_int', 'd', 'event'], dtype='object')

In [26]:
df_daily.isna().sum()

date              0
weekday           0
weekday_int       0
d                 0
event          1887
dtype: int64

In [27]:
display(df_daily.head())
df_daily.describe(include= 'all')

,date,weekday,weekday_int,d,event
0,2011-01-29,Saturday,1,d_1,NaN
1,2011-01-30,Sunday,2,d_2,NaN
2,2011-01-31,Monday,3,d_3,NaN
3,2011-02-01,Tuesday,4,d_4,NaN
4,2011-02-02,Wednesday,5,d_5,NaN


,date,weekday,weekday_int,d,event
count,1913,1913,1913.00,1913,26
unique,1913,7,NaN,1913,5
top,2011-01-29,Saturday,NaN,d_1,SuperBowl
freq,1,274,NaN,1,6
mean,NaN,NaN,4.00,NaN,NaN
std,NaN,NaN,2.00,NaN,NaN
min,NaN,NaN,1.00,NaN,NaN
25%,NaN,NaN,2.00,NaN,NaN
50%,NaN,NaN,4.00,NaN,NaN
75%,NaN,NaN,6.00,NaN,NaN


In [28]:
df_daily.dtypes

date           object
weekday        object
weekday_int      int8
d              object
event          object
dtype: object

In [29]:
df_daily['date'] = pd.to_datetime( df_daily['date'])
df_daily = df_daily.sort_values(by = 'date') 
# como esta ordenado luego me tiene que funcionar directamente lo de comprobar si esta completo

In [30]:
print(df_daily['date'].min())
print(df_daily['date'].max())
print(df_daily['date'].max() - df_daily['date'].min())

2011-01-29 00:00:00
2016-04-24 00:00:00
1912 days 00:00:00


El calendario tiene 1.913 días —> comprobamos que no falta ningún día.

In [31]:
def comprobar_fechas(c):
    inicio = c.min()
    fin = c.max()
    completo = pd.date_range(inicio, fin)
    print((completo == c).all())

comprobar_fechas(df_daily['date'])

True


In [32]:
df_daily['event'].value_counts(dropna= False)


event
NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: count, dtype: int64

La gran mayoría de días no tienen ningún evento registrado — solo 26 días de los 1.913 tienen evento.

A lo mejor podríamos incorporar más información al calendario, 
como por ejemplo si llovío o el tiempo, pero para más adelante cuando tengamos un df bien construido

In [33]:
display(df_daily[df_daily['event'] == 'Ramadan starts'])
display(df_daily[df_daily['event'] == 'SuperBowl'])
display(df_daily[df_daily['event'] == 'Thanksgiving'])
display(df_daily[df_daily['event'] == 'NewYear'])
display(df_daily[df_daily['event'] == 'Easter'])


,date,weekday,weekday_int,d,event
184,2011-08-01,Monday,3,d_185,Ramadan starts
538,2012-07-20,Friday,7,d_539,Ramadan starts
892,2013-07-09,Tuesday,4,d_893,Ramadan starts
1247,2014-06-29,Sunday,2,d_1248,Ramadan starts
1601,2015-06-18,Thursday,6,d_1602,Ramadan starts


,date,weekday,weekday_int,d,event
8,2011-02-06,Sunday,2,d_9,SuperBowl
372,2012-02-05,Sunday,2,d_373,SuperBowl
736,2013-02-03,Sunday,2,d_737,SuperBowl
1100,2014-02-02,Sunday,2,d_1101,SuperBowl
1464,2015-02-01,Sunday,2,d_1465,SuperBowl
1835,2016-02-07,Sunday,2,d_1836,SuperBowl


,date,weekday,weekday_int,d,event
299,2011-11-24,Thursday,6,d_300,Thanksgiving
663,2012-11-22,Thursday,6,d_664,Thanksgiving
1034,2013-11-28,Thursday,6,d_1035,Thanksgiving
1398,2014-11-27,Thursday,6,d_1399,Thanksgiving
1762,2015-11-26,Thursday,6,d_1763,Thanksgiving


,date,weekday,weekday_int,d,event
337,2012-01-01,Sunday,2,d_338,NewYear
703,2013-01-01,Tuesday,4,d_704,NewYear
1068,2014-01-01,Wednesday,5,d_1069,NewYear
1433,2015-01-01,Thursday,6,d_1434,NewYear
1798,2016-01-01,Friday,7,d_1799,NewYear


,date,weekday,weekday_int,d,event
435,2012-04-08,Sunday,2,d_436,Easter
792,2013-03-31,Sunday,2,d_793,Easter
1177,2014-04-20,Sunday,2,d_1178,Easter
1527,2015-04-05,Sunday,2,d_1528,Easter
1884,2016-03-27,Sunday,2,d_1885,Easter


Aquí lo que podemos apreciar es que tenemos una fecha por año y que no siempre coincide la fecha 
en la cae el evento todos los años,excepto en el caso de Año Nuevo.

Tenemos 5 registros de cada uno menos de la superbowl porque al ser en febrero lo tenemos tanto en la parte de 2011 
(a partir de febrero prácticamente) como en la parte de 2016 (hasta marzo)

Cada evento aparece una vez por año excepto SuperBowl, que al caer en febrero 
aparece tanto en 2011 como en 2016 al ser los extremos del dataset.

In [34]:
(df_daily['d'].str[2:].astype('int') == pd.Series(range(1,1914))).all()
# la columna d son los numeros normales ordenados y no falta ninguno.

np.True_

In [35]:
df_daily.groupby('weekday')['weekday_int'].value_counts()

weekday    weekday_int
Friday     7              273
Monday     3              273
Saturday   1              274
Sunday     2              274
Thursday   6              273
Tuesday    4              273
Wednesday  5              273
Name: count, dtype: int64

Vemos que es información redundante, por lo que eliminamos la columna 'weelday_int':

In [36]:
df_daily = df_daily.drop(columns=['weekday_int'])

In [37]:
df_daily['yearweek'] = df_daily['date'].dt.isocalendar().year.astype(str) + \
                       df_daily['date'].dt.isocalendar().week.astype(str).str.zfill(2)
df_daily['yearweek'] = df_daily['yearweek'].astype(np.float32)

df_daily

,date,weekday,d,event,yearweek
0,2011-01-29,Saturday,d_1,NaN,201104.00
1,2011-01-30,Sunday,d_2,NaN,201104.00
2,2011-01-31,Monday,d_3,NaN,201105.00
3,2011-02-01,Tuesday,d_4,NaN,201105.00
4,2011-02-02,Wednesday,d_5,NaN,201105.00
...,...,...,...,...,...
1908,2016-04-20,Wednesday,d_1909,NaN,201616.00
1909,2016-04-21,Thursday,d_1910,NaN,201616.00
1910,2016-04-22,Friday,d_1911,NaN,201616.00
1911,2016-04-23,Saturday,d_1912,NaN,201616.00


In [38]:
us_holidays = holidays.US(years=range(2011, 2017))

# Días festivos que cubre la librería en nuestro rango de fechas
festivos_df = pd.DataFrame([{'date': fecha, 'holiday': nombre}
                            for fecha, nombre in us_holidays.items()
                            ]).sort_values('date')

In [39]:
print("Eventos que ya tenemos:")
print(df_daily['event'].value_counts(dropna=False))

print("\nEventos que añade la librería:")
print(festivos_df['holiday'].value_counts())

Eventos que ya tenemos:
event
NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: count, dtype: int64

Eventos que añade la librería:
holiday
New Year's Day                 6
Martin Luther King Jr. Day     6
Washington's Birthday          6
Memorial Day                   6
Independence Day               6
Labor Day                      6
Columbus Day                   6
Veterans Day                   6
Thanksgiving Day               6
Christmas Day                  6
Christmas Day (observed)       2
New Year's Day (observed)      1
Veterans Day (observed)        1
Independence Day (observed)    1
Name: count, dtype: int64


La librería incluye festivos que ya teníamos (***NewYear***, ***Thanksgiving***) y versiones 
***observed*** (días de sustitución cuando el festivo cae en fin de semana)


In [40]:
us_holidays = holidays.US(years=range(2011, 2017))

def get_holiday(fecha):
    nombre = us_holidays.get(fecha)
    if nombre and '(observed)' not in nombre:
        return nombre
    return None

mask = df_daily['event'].isna()                       # Para no sobreescribir los eventos que ya tenemos
df_daily.loc[mask, 'event'] = df_daily.loc[mask, 'date'].apply(get_holiday)

df_daily['event'].value_counts(dropna=False)

event
None                          1846
SuperBowl                        6
Washington's Birthday            6
Memorial Day                     5
Independence Day                 5
Ramadan starts                   5
Labor Day                        5
Columbus Day                     5
Veterans Day                     5
Thanksgiving                     5
Christmas Day                    5
NewYear                          5
Martin Luther King Jr. Day       5
Easter                           5
Name: count, dtype: int64

Añadimos festivos oficiales de EEUU con la librería `holidays`. 

La ***máscara isna()*** protege los eventos que ya teníamos, y descartamos los ***(observed)*** porque los 
supermercados siguen la fecha real del festivo, no el traslado administrativo.

## CRUZAMOS LOS DATASETS

Las tres tablas están separadas y necesitamos unirlas en un único `df`. Lo hacemos en tres pasos:

1. **Melt df_sales ->** transformamos las ventas de formato ancho (una columna por día) a largo 
(una fila por día). Resultado: ~58M filas.

2. **Merge 1: Ventas + Calendario ->** añadimos a cada registro de venta la fecha real, 
el día de la semana y el evento de ese día, usando la columna `d` como clave.

3. **Merge 2: + Precios ->** añadimos el precio de cada producto en cada tienda y semana, 
usando `item + store_code + yearweek` como clave. Si no hay precio registrado esa semana, 
el campo `sell_price` queda como NaN.

In [41]:
# Paso 1:
df_sales_melt = pd.melt(df_sales, id_vars=df_sales.columns[:7], var_name='d', value_name='sales')
del df_sales
gc.collect()

# Strings como categóricas para ahorrar memoria
for col in ['item', 'id', 'store_code', 'store', 'region', 'category', 'd']:    
    df_sales_melt[col] = df_sales_melt[col].astype('category')

for col in ['item', 'store_code']:
    df_prices[col] = df_prices[col].astype('category')
    
    
# Paso 2:
df = df_sales_melt.merge(df_daily, on='d', how='left')
del df_sales_melt
gc.collect()

# Paso 3:
df = df.merge(df_prices.drop(columns='category'), on=['item', 'store_code', 'yearweek'], how='left')
del df_prices
gc.collect()

0

In [42]:
def estacion(mes):
    if mes in [12, 1, 2]:
        return 'Invierno'
    elif mes in [3, 4, 5]:
        return 'Primavera'
    elif mes in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['season'] = df['date'].dt.month.apply(estacion)

In [43]:
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,None,201616.00,3.58,Primavera
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201616.00,2.98,Primavera
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201616.00,4.78,Primavera
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,None,201616.00,1.54,Primavera


## EXPLORACIÓN

In [44]:
muestra_producto_tienda = df [ df.id == 'SUPERMARKET_3_594_NYC_2'] # son las ventas de un producto en una tienda solamente
px.line(muestra_producto_tienda, x = 'date', y = 'sales')

In [45]:
muestra_producto = df[df.item == 'SUPERMARKET_3_594'].groupby('date').sales.sum().reset_index()
muestra_producto['date'] = pd.to_datetime(muestra_producto['date'])
fig = px.line(muestra_producto, x='date', y='sales')
eventos = df[df['event'].notna()]
eventos['date'] = pd.to_datetime(eventos['date'])
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [46]:
# Vamos a ver cuanto vende cada tienda
df.groupby(by = ['store_code','date'])['sales'].sum().reset_index()

,store_code,date,sales
0,BOS_1,2011-01-29,2556
1,BOS_1,2011-01-30,2687
2,BOS_1,2011-01-31,1822
3,BOS_1,2011-02-01,2258
4,BOS_1,2011-02-02,1694
...,...,...,...
19125,PHI_3,2016-04-20,3159
19126,PHI_3,2016-04-21,3226
19127,PHI_3,2016-04-22,3828
19128,PHI_3,2016-04-23,4686


In [47]:
fig = px.line(df.groupby(by = ['store_code','date'])['sales'].sum().reset_index(), x = 'date', y = 'sales', color = 'store_code')
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [48]:
df[(df.date.dt.month == 12) & (df.date.dt.day == 25) & (df.sales != 0)]

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season
10067559,SUPERMARKET_3_586_NYC_2,SUPERMARKET_3_586,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_331,6,2011-12-25,Sunday,Christmas Day,201151.00,1.78,Invierno
10067725,SUPERMARKET_3_755_NYC_2,SUPERMARKET_3_755,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,1.42,Invierno
10070428,SUPERMARKET_3_406_NYC_3,SUPERMARKET_3_406,SUPERMARKET,SUPERMARKET_3,Tribeca,NYC_3,New York,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,0.98,Invierno
10082362,SUPERMARKET_3_144_BOS_3,SUPERMARKET_3_144,SUPERMARKET,SUPERMARKET_3,Back_Bay,BOS_3,Boston,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,2.26,Invierno
10082444,SUPERMARKET_3_226_BOS_3,SUPERMARKET_3_226,SUPERMARKET,SUPERMARKET_3,Back_Bay,BOS_3,Boston,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,1.78,Invierno
10088542,SUPERMARKET_3_226_PHI_2,SUPERMARKET_3_226,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,1.78,Invierno
10088554,SUPERMARKET_3_238_PHI_2,SUPERMARKET_3_238,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,4.18,Invierno
10088577,SUPERMARKET_3_261_PHI_2,SUPERMARKET_3_261,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,Christmas Day,201151.00,5.38,Invierno
21226868,SUPERMARKET_3_555_NYC_2,SUPERMARKET_3_555,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_697,1,2012-12-25,Tuesday,Christmas Day,201252.00,1.90,Invierno
21226899,SUPERMARKET_3_586_NYC_2,SUPERMARKET_3_586,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_697,1,2012-12-25,Tuesday,Christmas Day,201252.00,1.90,Invierno


In [49]:
fig = px.bar(df.groupby(['category', 'store_code'])['sales'].sum().reset_index(), x='category', y='sales', color='store_code', barmode='stack')
fig.show()

In [50]:
ventas_dia = df.groupby(by = ['store_code', 'weekday'])['sales'].sum().reset_index()
ventas_dia = ventas_dia.pivot(index = 'weekday', columns = 'store_code', values = 'sales')
fig = px.imshow(ventas_dia, color_continuous_scale = 'viridis')
fig.show()

In [51]:
fig = px.box(df.groupby(['store_code', 'date'])['sales'].sum().reset_index(), x='store_code', y='sales', color='store_code')
fig.show()

## PRODUCTOS

### Top productos por ciudad

In [52]:
top_productos = df.groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
print("top productos en general:")
print(top_productos)

for region in ['New York', 'Boston', 'Philadelphia']:
    print(f"\nTop 10 en {region}:")
    top_region = df[df['region'] == region].groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
    print(top_region)

top productos en general:
item
SUPERMARKET_3_090    1002529
SUPERMARKET_3_586     920242
SUPERMARKET_3_252     565299
SUPERMARKET_3_555     491287
SUPERMARKET_3_714     396172
SUPERMARKET_3_587     396119
SUPERMARKET_3_694     390001
SUPERMARKET_3_226     363082
SUPERMARKET_3_202     295689
SUPERMARKET_3_723     284333
Name: sales, dtype: int64

Top 10 en New York:
item
SUPERMARKET_3_090    486138
SUPERMARKET_3_586    318050
SUPERMARKET_3_252    237172
SUPERMARKET_3_120    176446
SUPERMARKET_3_587    166065
SUPERMARKET_3_808    165778
SUPERMARKET_3_635    158166
SUPERMARKET_3_541    156589
SUPERMARKET_3_714    146000
SUPERMARKET_3_555    141146
Name: sales, dtype: int64

Top 10 en Boston:
item
SUPERMARKET_3_586    455411
SUPERMARKET_3_090    328034
SUPERMARKET_3_252    257079
SUPERMARKET_3_555    244685
SUPERMARKET_3_377    164976
SUPERMARKET_3_587    138710
SUPERMARKET_3_714    129772
SUPERMARKET_3_202    123263
SUPERMARKET_3_030    109496
SUPERMARKET_3_694    104541
Name: sales, dtyp

### Productos en declive

In [53]:
# Para eso vamos a agrupar por meses para ver cuales van disminuyendo
df['year_month'] = df['date'].dt.to_period('M')
ventas_mensuales = df.groupby(by = ['item', 'year_month']).sales.sum().reset_index()
ventas_mensuales['year_month'] = ventas_mensuales['year_month'].astype(str)
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,year_month
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno,2011-01
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno,2011-01
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno,2011-01
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno,2011-01
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201104.00,NaN,Invierno,2011-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,None,201616.00,3.58,Primavera,2016-04
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201616.00,2.98,Primavera,2016-04
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201616.00,4.78,Primavera,2016-04
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,None,201616.00,1.54,Primavera,2016-04


In [54]:
producto_ejemplo = ventas_mensuales['item'].iloc[120] # este por ejemplo tiene siempre picos en diciembre
data_ejemplo = ventas_mensuales[ventas_mensuales['item'] == producto_ejemplo]
fig = px.line(data_ejemplo, x='year_month', y='sales', title=f'Ventas mensuales: {producto_ejemplo}')
fig.show()

In [55]:
def calcular_tendencia_producto(df):
    try:
        if len(df) < 24:
            return np.nan
        resultado = seasonal_decompose(df['sales'], model = 'multiplicative', period=6)
        tendencia = resultado.trend.dropna() #quito el principio y el final pq se me quedan muchos nulos
        x = np.arange(len(tendencia))
        pendiente = np.polyfit(x, tendencia, 1)[0]
        return pendiente
    except:
        return np.nan


tendencias_productos = []
for item in ventas_mensuales['item'].unique():
    producto = ventas_mensuales[ventas_mensuales['item'] == item].sort_values('year_month')
    pendiente = calcular_tendencia_producto(producto)
    tendencias_productos.append({
        'item': item,
        'pendiente_tendencia': pendiente,
        'ventas_totales': producto['sales'].sum()
    })
df_tendencias = pd.DataFrame(tendencias_productos)
df_tendencias = df_tendencias.dropna()
df_tendencias = df_tendencias.sort_values('pendiente_tendencia')
print("productos con pendiente + negativa:")
print(df_tendencias.head(20))

productos con pendiente + negativa:
                   item  pendiente_tendencia  ventas_totales
2374  SUPERMARKET_3_150               -96.36          147532
2186  SUPERMARKET_2_360               -78.84          257119
2779  SUPERMARKET_3_555               -70.33          491287
2810  SUPERMARKET_3_586               -63.21          920242
2859  SUPERMARKET_3_635               -56.39          282134
1954  SUPERMARKET_2_128               -55.13          149465
1848  SUPERMARKET_2_021               -43.94          121679
2378  SUPERMARKET_3_154               -43.14          101583
2670  SUPERMARKET_3_446               -41.87           96285
2692  SUPERMARKET_3_468               -39.32           85612
2304  SUPERMARKET_3_080               -37.80          262650
2600  SUPERMARKET_3_376               -36.20          228551
2102  SUPERMARKET_2_276               -34.87          158887
3029  SUPERMARKET_3_808               -32.78          281879
2543  SUPERMARKET_3_319               -29.36     

In [56]:
productos_declive = df_tendencias.head(10)
fig = make_subplots(rows=5, cols=2, subplot_titles=[f"{prod}" for prod, pend in zip(productos_declive['item'], productos_declive['pendiente_tendencia'])])

for idx, prod in enumerate(productos_declive['item']):
    datos_prod = ventas_mensuales[ventas_mensuales['item'] == prod].sort_values('year_month')
    fig.add_trace(go.Scatter(x=datos_prod['year_month'], y=datos_prod['sales'], mode='lines+markers', name=prod, showlegend=False), row=idx//2+1, col=idx%2+1)

fig.update_layout(height=1200, title_text="Top 10 Productos en Declive")
fig.show()

### productos con mas diferencia de precios entre tiendas

In [57]:
precios = df.groupby('item')['sell_price'].describe() # tenemos los 3049 productos, vamos a quedarnos con los q presentasen mayor varianza en los precios
precios['rango'] = precios['max'] - precios['min']
precios.sort_values('rango', ascending=False).head(20)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19019.00,16.06,5.97,4.07,15.57,15.57,15.59,134.15,130.07
HOME_&_GARDEN_2_466,18382.00,9.41,5.44,7.46,8.71,8.71,8.71,65.78,58.31
HOME_&_GARDEN_2_178,18361.00,8.74,1.59,3.75,8.71,8.71,8.71,55.45,51.70
HOME_&_GARDEN_2_250,19012.00,8.80,1.38,4.20,8.70,8.70,8.71,42.72,38.52
HOME_&_GARDEN_2_514,19019.00,23.46,1.15,1.25,22.42,23.71,23.71,26.21,24.96
ACCESORIES_1_342,13818.00,26.29,1.23,2.66,25.23,27.11,27.11,27.11,24.45
HOME_&_GARDEN_1_469,18844.00,22.47,0.88,1.25,22.46,22.46,22.46,25.58,24.33
ACCESORIES_1_257,18340.00,22.35,0.52,10.59,22.29,22.29,22.29,34.31,23.73
HOME_&_GARDEN_1_342,7917.00,22.39,0.92,1.25,22.40,22.46,22.46,22.46,21.21


In [58]:
precios.sort_values(by = 'std', ascending = False).head(10)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19019.00,16.06,5.97,4.07,15.57,15.57,15.59,134.15,130.07
HOME_&_GARDEN_2_466,18382.00,9.41,5.44,7.46,8.71,8.71,8.71,65.78,58.31
HOME_&_GARDEN_2_292,18956.00,15.80,3.15,12.46,12.46,18.71,18.71,19.96,7.50
ACCESORIES_1_060,13783.00,39.49,2.07,34.58,39.85,39.86,41.20,41.20,6.62
ACCESORIES_1_361,13853.00,39.49,2.07,35.88,39.85,39.86,41.20,41.20,5.32
ACCESORIES_1_225,13853.00,39.49,2.06,35.88,39.85,39.86,41.20,41.20,5.32
HOME_&_GARDEN_1_259,19040.00,12.14,2.01,7.49,9.35,13.71,13.71,14.40,6.91
ACCESORIES_1_311,17843.00,20.97,1.99,17.29,19.50,19.50,23.65,23.91,6.62
HOME_&_GARDEN_2_446,18984.00,31.82,1.85,18.74,31.15,32.46,32.46,35.00,16.26


1. Rango de precios (max - min): mide la diferencia entre la tienda más cara y la más barata para cada producto. Un rango alto indica que el mismo producto se vende a precios muy diferentes según la ubicación.
2. Desviación estándar: mide cuánto varían los precios en general. Mientras que el rango solo captura los extremos, la desviación estándar considera todos los precios y detecta si la variación es generalizada o si es solo una tienda atípica.

El rango solo mira extremos y la desviación penaliza más cuando MUCHOS precios están dispersos. Yo le preguntaría a Dani porque no sé cual nos puede interesar más

### productos estacionales

In [59]:
ventas_estacion = df.groupby(['item', 'season'])['sales'].sum().reset_index()
ventas_estacion['total_producto'] = ventas_estacion.groupby('item')['sales'].transform('sum')
ventas_estacion['pct'] = (ventas_estacion['sales'] / ventas_estacion['total_producto'] * 100).round(1)
# ventas_estacion
resumen_estaciones = ventas_estacion.pivot(index='item', columns='season', values='pct')
resumen_estaciones['max_season'] = resumen_estaciones.max(axis=1)
productos_estacionales = resumen_estaciones.sort_values('max_season', ascending=False).head(20)
print("Productos más estacionales (concentrados en una época):")
print(productos_estacionales)

Productos más estacionales (concentrados en una época):
season               Invierno  Otoño  Primavera  Verano  max_season
item                                                               
HOME_&_GARDEN_1_297      4.50  13.30      20.10   62.10       62.10
HOME_&_GARDEN_2_162      0.30  23.90      16.00   59.80       59.80
HOME_&_GARDEN_2_340      2.80   5.10      32.60   59.60       59.60
HOME_&_GARDEN_1_189      4.00  18.40      19.60   58.00       58.00
HOME_&_GARDEN_2_022     11.00   6.40      57.90   24.60       57.90
HOME_&_GARDEN_1_049      0.10   3.80      38.50   57.60       57.60
HOME_&_GARDEN_2_449      4.30   9.70      28.40   57.60       57.60
HOME_&_GARDEN_2_289      2.70  15.90      24.00   57.40       57.40
HOME_&_GARDEN_2_344      6.10  17.80      19.70   56.40       56.40
HOME_&_GARDEN_2_080      6.30   7.80      29.80   56.10       56.10
HOME_&_GARDEN_2_477      4.60  19.70      21.10   54.60       54.60
SUPERMARKET_3_350       14.70  13.60      54.10   17.50     

In [60]:
top_estacionales = productos_estacionales.head(5)
fig = make_subplots(rows=3, cols=2, subplot_titles=[f"{prod}" for prod in top_estacionales.index[:5]])

for idx, prod in enumerate(top_estacionales.index[:5]):
    datos_prod = df[df['item'] == prod].groupby('date')['sales'].sum().reset_index()
    fig.add_trace(go.Scatter(x=datos_prod['date'], y=datos_prod['sales'], mode='lines', name=prod, showlegend=False), row=idx//2+1, col=idx%2+1)

fig.update_layout(height=900, title_text="Evolución Temporal - Top 5 Productos Estacionales")
fig.show()